# EMD1 RNA methylation and reader effects

Run all cells with the default settings to reproduce Figures 1, 2, S3 and S4. The simulation compares transcript responses to arsenic exposure, FTO inhibition and METTL3 knockdown. Installation instructions are in the [README](README.md).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "emd1_simulation").is_dir() or not (ROOT / "data" / "emd1.sqlite").is_file():
    raise FileNotFoundError("Start Jupyter in the EMD1 directory and open the notebook there.")
sys.path.insert(0, str(ROOT / "src"))
print("EMD1 directory:", ROOT)


## Run settings

Use the defaults for the manuscript figures. Set a new `RUN_NAME` to keep an exploratory run. Reusing a name replaces its generated files and removes outputs that have been switched off.

`PARAMETER_OVERRIDES` accepts independent model parameters, for example `{"k_E": 2.0}`. The model derives baseline synthesis rates and resting ROS. Occupancy fractions (`M0_*`) and `w_aox_fto` range from 0.05 to 0.90. The ALKBH5 inhibition amplitude `a_B` ranges from 0 to 1. The delayed intervention takes effect only when the run extends beyond nominal day 30.

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from dataclasses import asdict
from IPython.display import Image, Markdown, display
from emd1_simulation.processing.store import DATABASE_PATH, read_table
from emd1_simulation.simulation.settings import SimulationSettings, prepare_parameters
from emd1_simulation.workflow import run_workflow
from emd1_simulation.reporting.tables import summary_tables, trajectory_table
from emd1_simulation.reporting.provenance import RELEASE_VERSION

# Simulation
RUN_NAME = "reproduced"
N_ENSEMBLE = 400
T_END_DAYS = 45.0
N_TIME_POINTS = 451
ENSEMBLE_SEED = 20260822
COUNTERFACTUAL_SEED = 20260825
PARAMETER_OVERRIDES = {}

# Outputs
MAKE_FIGURE = True
EXPORT_TABLES = True
SELECTED_TABLES = ["endpoint_folds", "arsenic_sensitivity_endpoints",
                   "population_diagnostics", "model_checks", "scientific_comparisons"]

if not isinstance(RUN_NAME, str) or not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME must be a single folder name, for example 'lower_ros_gain'.")
parameters = prepare_parameters(PARAMETER_OVERRIDES)
settings = SimulationSettings(
    t_end_days=T_END_DAYS, n_time_points=N_TIME_POINTS, n_ensemble=N_ENSEMBLE,
    ensemble_seed=ENSEMBLE_SEED, counterfactual_seed=COUNTERFACTUAL_SEED)
OUTDIR = ROOT / "outputs" / RUN_NAME
OUTDIR.mkdir(parents=True, exist_ok=True)
print("Run folder:", OUTDIR)
print(f"Horizon: {T_END_DAYS:g} nominal model days | grid points: {N_TIME_POINTS} | draws: {N_ENSEMBLE}")

## Source measurements

The SQLite database supplies published measurements for comparisons and figure overlays. Chase half-lives are measured in experimental hours. The model uses nominal days. DDB2 and XPC enrichment is summarized over the first three exons. The corresponding background percentiles describe variation across transcripts. Assay definitions, citations and coordinate conventions are in the [data notes](data/THIRD_PARTY.md).

In [ ]:
display(pd.DataFrame(
    {"table": name, "rows": len(read_table(name))}
    for name in ("chase_decay_constants", "digitised_endpoints", "repair_edge_summary")
))

## Run the simulation

The model produces nominal trajectories, a parameter-sensitivity ensemble and repair scenarios. The numerical summary records the settings, input hashes and software versions. The run log reports how many model checks passed. The `model_checks` table gives each result.

In [ ]:
from contextlib import redirect_stdout, redirect_stderr
from time import perf_counter

log_path = OUTDIR / "emd1_reproduction.log"
started = perf_counter()
with log_path.open("w", encoding="utf-8") as log:
    with redirect_stdout(log), redirect_stderr(log):
        experiment = run_workflow(
            OUTDIR, settings, params=parameters,
            make_figure=MAKE_FIGURE, make_tables=EXPORT_TABLES)
status = experiment.exit_code
if status not in (0, 1):
    raise RuntimeError(f"Run returned status {status}; inspect {log_path}")
summary = experiment.summary
simulated = experiment.simulation
tables = summary_tables(summary)
tables["nominal_trajectories"] = trajectory_table(simulated.nominal, simulated.time)
tables["relative_trajectories"] = trajectory_table(simulated.nominal, simulated.time, relative=True)
tables["source_protocol_trajectories"] = trajectory_table(simulated.source_protocols, simulated.time)
print(f"Completed in {perf_counter() - started:.1f} s")
print(f"Internal checks: {summary['model_checks_passed']}/{len(summary['model_checks'])}")
print(f"Ensemble: {summary['ensemble_draws_ok']}/{N_ENSEMBLE} accepted draws")
print(f"Counterfactual: {summary['repair_edge']['counterfactual_draws_ok']}/{N_ENSEMBLE} accepted draws")
if status == 1:
    print("See the model_checks table for the checks that failed.")
print("Full log:", log_path)

## Figures

Shading shows the 5th–95th percentiles across parameter draws. Figure S4 also shows a reference range from the experimental transcripts. Each caption identifies the curves, conditions and units.

In [ ]:
if MAKE_FIGURE:
    from emd1_simulation.reporting.figure import FIGURE_GROUPS

    for group in FIGURE_GROUPS.values():
        stem = group["stem"]
        display(Image(filename=str(OUTDIR / f"{stem}.png"), width=1100))
        display(Markdown((OUTDIR / f"{stem}_legend.md").read_text()))

## Numerical results

`SELECTED_TABLES` controls the previews below. The `tables` dictionary contains all data frames, including native and control-normalized trajectories. Relative population expansion (`lnP`) is a log₂ fold change. Relative net growth is a rate difference per nominal model day. The observable dictionary gives the units for each quantity.

In [ ]:
for name in SELECTED_TABLES:
    if name not in tables:
        raise ValueError(f"Unknown preview table {name!r}. Choose from {list(tables)}")
    display(Markdown(f"### {name.replace('_', ' ').capitalize()}"))
    frame = tables[name]
    if name == "endpoint_folds":
        display(frame.pivot(index="condition", columns="observable", values="value").round(4))
    elif name == "population_diagnostics":
        display(frame.pivot(index="condition", columns="metric", values="value"))
    else:
        display(frame)

## Reproducibility checks

These checks verify input hashes, parameter values and the scientific assumptions recorded in the summary. For the manuscript settings they also compare selected endpoints and the median four-state methylation response with the reported values.

In [ ]:
metadata = summary["reproducibility"]
assert metadata["input_database_sha256"] == hashlib.sha256(DATABASE_PATH.read_bytes()).hexdigest()
assert metadata["parameters"] == asdict(parameters)
assert metadata["release_version"] == RELEASE_VERSION
assert summary["schema_version"] == 3
assert summary["scientific_evidence_audit"]["quantitatively_validated"] is False
assert summary["independent_biological_validation_performed"] is False
assert metadata["source_status"]["sources"]["nedd4l_exon1"]["quantitative_target_fold"] is None
assert not any(r["directly_measured"] for r in metadata["parameter_provenance"].values())
for filename, digest in metadata["scientific_metadata_sha256"].items():
    assert digest == hashlib.sha256((ROOT / "data" / filename).read_bytes()).hexdigest()
if metadata["canonical_model_and_grid"]:
    assert summary["model_checks_passed"] == len(summary["model_checks"])
    # Rounded nominal arsenic values reported for a 45-day simulation.
    endpoints = summary["endpoints_fold_vs_control"]["as"]
    np.testing.assert_allclose([endpoints["R"], endpoints["M_aox"]], [3.36, 1.24], atol=0.005, rtol=0)
if metadata["canonical_ensemble"]:
    assert summary["ensemble_draws_ok"] == N_ENSEMBLE
    assert summary["repair_edge"]["counterfactual_draws_ok"] == N_ENSEMBLE
    np.testing.assert_allclose(summary["ensemble"]["M_mean4"]["p50"], 0.84, atol=0.005, rtol=0)
for output in experiment.paths + [log_path]:
    assert output.is_file() and output.stat().st_size > 0, output
print(f"Checks passed. Results are in {OUTDIR.relative_to(ROOT)}/")